In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import easyocr
import whisper
import gradio as gr

print("Loading OCR model...")
reader = easyocr.Reader(['en'])

print("Loading translation model...")
model_name = "Helsinki-NLP/opus-mt-en-ur"
tokenizer = AutoTokenizer.from_pretrained(model_name)
translation_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print("Loading Whisper model...")
voice_model = whisper.load_model("base")

print("All models loaded! Launching app...")

def translate_text(text):
    if not text.strip(): return ""
    inputs = tokenizer(text, return_tensors="pt")
    translated_tokens = translation_model.generate(**inputs, max_length=512)
    return tokenizer.decode(translated_tokens[0], skip_special_tokens=True)

def translate_image(img):
    results = reader.readtext(img)
    extracted_text = " ".join([res[1] for res in results])
    return translate_text(extracted_text)

def translate_voice(audio_path):
    result = voice_model.transcribe(audio_path)
    return translate_text(result['text'])

with gr.Blocks() as demo:
    gr.Markdown("# 🇵🇰 AI English to Urdu Translator")

    with gr.Tab("Text Translation"):
        txt_input = gr.Textbox(label="Enter English")
        txt_output = gr.Textbox(label="Urdu Result")
        btn_txt = gr.Button("Translate Text")
        btn_txt.click(translate_text, inputs=txt_input, outputs=txt_output)

    with gr.Tab("Image Translation"):
        img_input = gr.Image(type="filepath", label="Upload Image/Screenshot")
        img_output = gr.Textbox(label="Urdu Result")
        btn_img = gr.Button("Extract & Translate")
        btn_img.click(translate_image, inputs=img_input, outputs=img_output)

    with gr.Tab("Voice Translation"):
        aud_input = gr.Audio(type="filepath", label="Record or Upload Audio")
        aud_output = gr.Textbox(label="Urdu Result")
        btn_aud = gr.Button("Transcribe & Translate")
        btn_aud.click(translate_voice, inputs=aud_input, outputs=aud_output)

demo.launch(debug=True)

Loading OCR model...
Loading translation model...


/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Loading Whisper model...
All models loaded! Launching app...
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://fa9ead93c9c875bff4.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
!pip install easyocr openai-whisper gradio sacremoses -q
!pip install transformers sentencepiece -q

import torch
from transformers import pipeline
import easyocr
import whisper
import gradio as gr

print("Loading OCR...")
reader = easyocr.Reader(['en'])

print("Loading translation model (NLLB)...")
translator = pipeline(
    "translation",
    model="facebook/nllb-200-distilled-600M",
    src_lang="eng_Latn",
    tgt_lang="urd_Arab",
    max_length=512
)

print("Loading Whisper...")
voice_model = whisper.load_model("small")

print("All loaded!")

def translate_text(text):
    if not text.strip(): return ""
    result = translator(text)
    return result[0]['translation_text']

def translate_image(img):
    results = reader.readtext(img)
    extracted = " ".join([r[1] for r in results])
    return extracted, translate_text(extracted)

def translate_voice(audio_path):
    result = voice_model.transcribe(audio_path, language="en")
    transcribed = result['text']
    translated = translate_text(transcribed)
    return f"📝 Original: {transcribed}\n\n🔤 Urdu: {translated}"

with gr.Blocks(title="Urdu Translator") as demo:
    gr.Markdown("# 🇵🇰 AI English → Urdu Translator (Improved)")

    with gr.Tab("📝 Text"):
        t_in = gr.Textbox(label="English Text", lines=3)
        t_out = gr.Textbox(label="Urdu", lines=3)
        gr.Button("Translate").click(translate_text, t_in, t_out)

    with gr.Tab("🖼️ Image"):
        i_in = gr.Image(type="filepath", label="Upload Image")
        with gr.Row():
            i_extracted = gr.Textbox(label="Extracted Text")
            i_out = gr.Textbox(label="Urdu")
        gr.Button("Extract & Translate").click(translate_image, i_in, [i_extracted, i_out])

    with gr.Tab("🎙️ Voice"):
        a_in = gr.Audio(type="filepath", label="Record or Upload")
        a_out = gr.Textbox(label="Result", lines=4)
        gr.Button("Transcribe & Translate").click(translate_voice, a_in, a_out)

demo.launch(share=True, debug=True)